# 🎬 Colab CogVideoX Server

Run CogVideoX-5B-I2V (image-to-video) on T4 GPU with fp8 optimization.

**Just click Runtime > Run all** — everything runs automatically.

This is a modern video model that understands actions like "washing hair", "walking", etc.


## Step 1: Check GPU

In [ ]:
!nvidia-smi


## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
ROOT = "/content/drive/MyDrive/ComfyUI"
for d in ["models/checkpoints", "input", "output"]:
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)
print("Drive ready:", ROOT)


## Step 3: Install ComfyUI

In [ ]:
import os
COMFY = "/content/workspace/ComfyUI"
if not os.path.exists(COMFY):
    !git clone --depth=1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}
!cd {COMFY} && pip install -r requirements.txt -q --no-cache-dir
print("ComfyUI installed")


## Step 4: Install CogVideoXWrapper + KJNodes

In [ ]:
import os
N = f"{COMFY}/custom_nodes"
for name, url in [
    ("ComfyUI-CogVideoXWrapper", "https://github.com/kijai/ComfyUI-CogVideoXWrapper.git"),
    ("ComfyUI-KJNodes", "https://github.com/kijai/ComfyUI-KJNodes.git"),
    ("ComfyUI-VideoHelperSuite", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),
]:
    p = f"{N}/{name}"
    if not os.path.exists(p):
        !git clone --depth=1 {url} {p}
        print(f"Installed {name}")
    else:
        print(f"Have {name}")

# Install deps
!pip install -q -r {N}/ComfyUI-CogVideoXWrapper/requirements.txt
print("All nodes ready")


## Step 5: Download T5 Text Encoder

In [ ]:
import os
CLIP_DIR = f"{COMFY}/models/clip/t5"
os.makedirs(CLIP_DIR, exist_ok=True)
T5_PATH = f"{CLIP_DIR}/google_t5-v1_1-xxl_encoderonly-fp8_e4m3fn.safetensors"
if not os.path.exists(T5_PATH):
    !wget -q --show-progress \
      "https://huggingface.co/Comfy-Org/HuggingFace-Download/resolve/main/models/t5/google_t5-v1_1-xxl_encoderonly-fp8_e4m3fn.safetensors" \
      -O "{T5_PATH}"
    print("T5 encoder downloaded")
else:
    print("T5 encoder already exists")


## Step 6: Link I/O to Drive

In [ ]:
import os
for src, name in [("/content/drive/MyDrive/ComfyUI/output","output"),
                  ("/content/drive/MyDrive/ComfyUI/input","input")]:
    dst = f"{COMFY}/{name}"
    if os.path.islink(dst) or os.path.isdir(dst):
        !rm -rf {dst}
    !ln -s {src} {dst}
print("I/O linked to Drive")


## Step 7: Start ComfyUI + Cloudflare Tunnel

In [ ]:
import subprocess, time, os, re, psutil, socket

def kill_process_on_port(port):
    for proc in psutil.process_iter(["pid", "name", "connections"]):
        try:
            for conn in proc.connections(kind="inet"):
                if conn.laddr.port == port:
                    print(f"Killing process {proc.pid} ({proc.name()}) on port {port}")
                    proc.terminate()
                    proc.wait(timeout=5)
        except Exception:
            pass

if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

os.chdir(COMFY)
kill_process_on_port(8188)
print("Starting ComfyUI (CogVideoX mode)...")
!rm -f /content/comfyui.log
os.system("nohup python main.py --listen 0.0.0.0 --port 8188 > /content/comfyui.log 2>&1 &")

def wait_for_port(port, timeout=90):
    start_time = time.time()
    while time.time() - start_time < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(2)
    return False

if wait_for_port(8188):
    print("ComfyUI active on port 8188.")
    print("Starting Cloudflared tunnel...")
    tunnel_proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    COMFY_URL = None
    for _ in range(30):
        line = tunnel_proc.stderr.readline().decode(errors="replace").strip()
        m = re.search(r"https://[\w.-]+\.trycloudflare\.com", line)
        if m:
            COMFY_URL = m.group(0)
            break
        time.sleep(1)
    if COMFY_URL:
        print(f"Tunnel URL: {COMFY_URL}")
        url_file = "/content/drive/MyDrive/ComfyUI/tunnel_url.txt"
        with open(url_file, "w") as f:
            f.write(COMFY_URL)
        print(f"URL saved to Drive: {url_file}")
    else:
        print("ERROR: Failed to get tunnel URL")
else:
    print("ERROR: ComfyUI failed to start")


## Step 8: Server Ready - Waiting for Tasks
Run `python gen_cogvideo.py --url <TUNNEL_URL> --image input.png` from your local machine.


In [ ]:
import time
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Server stopped.")
